# Wheather forcasting 

In [156]:
import numpy as np
import pandas as pd
import requests

In [170]:
def historical_data(lat,long,start_date,end_date):
    url1 = "https://archive-api.open-meteo.com/v1/archive"
    params1 = {
        "latitude" : lat,
        "longitude" : long,    
        "start_date" : start_date,
        "end_date" : end_date,
        "daily" : "temperature_2m_mean,relative_humidity_2m_mean,wind_speed_10m_mean",
        "timezone" : "auto"
    }
    url2 = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params2 = {
        "latitude" : lat,
        "longitude" : long,    
        "start_date" : start_date,
        "end_date" : end_date,
        "hourly" : "pm2_5,pm10",
        "timezone" : "auto"
    }
    weather = requests.get(url1,params1).json()
    air = requests.get(url2,params2).json()
    df1=pd.DataFrame(weather["daily"])
    df1["time"]=pd.to_datetime(df1["time"])
    df1["time"]=df1["time"].dt.date
    df2=pd.DataFrame(air["hourly"])
    df2["time"]=pd.to_datetime(df2["time"])
    df2["time"]=df2["time"].dt.date
    df2=df2.groupby("time")[["pm2_5","pm10"]].mean().reset_index()
    df=pd.merge(df1,df2)
    df.columns=["date","temperature","humidity","wind_speed","pm2.5","pm10"]
    df["date"]=pd.to_datetime(df["date"])
    return df


In [231]:
df = historical_data(
    28.7041, 77.1025,
    "2022-01-01",
    "2025-12-27"
)
df.dropna()

,date,temperature,humidity,wind_speed,pm2.5,pm10
215,2022-08-04,28.5,83,8.3,43.594737,63.178947
216,2022-08-05,27.9,85,9.3,43.929167,63.750000
217,2022-08-06,28.7,83,4.3,73.745833,107.050000
218,2022-08-07,27.9,85,8.2,42.170833,61.029167
219,2022-08-08,29.3,79,6.0,52.733333,75.850000
...,...,...,...,...,...,...
1452,2025-12-23,15.7,83,8.2,131.420833,134.104167
1453,2025-12-24,15.3,72,9.4,57.645833,60.050000
1454,2025-12-25,13.8,70,4.7,92.320833,94.820833
1455,2025-12-26,14.0,75,4.0,129.504167,131.233333


In [232]:
#aqi calculation
df2_5= pd.DataFrame({
    "BPlo":[0,30,60,90,120,250],
    "BPhi":[30,60,90,120,250,500],
    "Ilo":[0,51,101,201,301,401],
    "Ihi":[50,100,200,300,400,500]
})
df10= pd.DataFrame({
    "BPlo":[0,50,100,250,350,450],
    "BPhi":[50,100,250,350,450,600],
    "Ilo":[0,51,101,201,301,401],
    "Ihi":[50,100,200,300,400,500]
})

In [249]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1457 entries, 0 to 1456
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         1457 non-null   datetime64[ns]
 1   temperature  1457 non-null   float64       
 2   humidity     1457 non-null   int64         
 3   wind_speed   1457 non-null   float64       
 4   pm2.5        1242 non-null   float64       
 5   pm10         1242 non-null   float64       
 6   AQI          1242 non-null   float64       
 7   pm2.5_lag1   1241 non-null   float64       
 8   pm10_lag1    1241 non-null   float64       
 9   pm2.5_lag2   1240 non-null   float64       
 10  pm10_lag2    1240 non-null   float64       
 11  pm2.5_lag3   1239 non-null   float64       
 12  pm10_lag3    1239 non-null   float64       
 13  pm2.5_roll7  1236 non-null   float64       
 14  pm10_roll7   1236 non-null   float64       
dtypes: datetime64[ns](1), float64(13), int64(1)
memory usag

In [234]:
def aqi(c,dfp):
    if pd.isna(c):
        return np.nan
    row = dfp[(dfp["BPlo"] <= c) & (c <= dfp["BPhi"])]
    if row.empty:
        if c > dfp["BPhi"].max():
            return dfp["Ihi"].max()
        else:
            return np.nan
    Ihi=row["Ihi"].iloc[0]
    Ilo=row["Ilo"].iloc[0]
    BPhi=row["BPhi"].iloc[0]
    BPlo=row["BPlo"].iloc[0]
    AQI=((Ihi-Ilo)/(BPhi-BPlo))*(c-BPlo)+Ilo
    return AQI

In [235]:
df["AQI2.5"] = df["pm2.5"].apply(lambda x: aqi(x, df2_5))
df["AQI10"]  = df["pm10"].apply(lambda x: aqi(x, df10))
df["AQI"]    = df[["AQI2.5", "AQI10"]].max(axis=1)

In [236]:
df.drop(["AQI2.5","AQI10"],axis=1,inplace=True)

In [237]:
df

,date,temperature,humidity,wind_speed,pm2.5,pm10,AQI
0,2022-01-01,10.4,79,7.8,NaN,NaN,NaN
1,2022-01-02,11.3,80,7.7,NaN,NaN,NaN
2,2022-01-03,11.8,79,8.2,NaN,NaN,NaN
3,2022-01-04,14.1,73,6.1,NaN,NaN,NaN
4,2022-01-05,13.3,89,9.8,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1452,2025-12-23,15.7,83,8.2,131.420833,134.104167,309.697404
1453,2025-12-24,15.3,72,9.4,57.645833,60.050000,96.154861
1454,2025-12-25,13.8,70,4.7,92.320833,94.820833,208.658750
1455,2025-12-26,14.0,75,4.0,129.504167,131.233333,308.237788


# Random_Forest Regressor

In [238]:
df.sort_values("date").reset_index(drop=True)

,date,temperature,humidity,wind_speed,pm2.5,pm10,AQI
0,2022-01-01,10.4,79,7.8,NaN,NaN,NaN
1,2022-01-02,11.3,80,7.7,NaN,NaN,NaN
2,2022-01-03,11.8,79,8.2,NaN,NaN,NaN
3,2022-01-04,14.1,73,6.1,NaN,NaN,NaN
4,2022-01-05,13.3,89,9.8,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1452,2025-12-23,15.7,83,8.2,131.420833,134.104167,309.697404
1453,2025-12-24,15.3,72,9.4,57.645833,60.050000,96.154861
1454,2025-12-25,13.8,70,4.7,92.320833,94.820833,208.658750
1455,2025-12-26,14.0,75,4.0,129.504167,131.233333,308.237788


In [239]:
for lag in [1,2,3]:
    df[f"pm2.5_lag{lag}"]=df["pm2.5"].shift(lag)
    df[f"pm10_lag{lag}"]=df["pm10"].shift(lag)
df["pm2.5_roll7"]=df["pm2.5"].rolling(7).mean()
df["pm10_roll7"]=df["pm10"].rolling(7).mean()
dfml=df.dropna().reset_index(drop=True)

In [240]:
dfml

,date,temperature,humidity,wind_speed,pm2.5,pm10,AQI,pm2.5_lag1,pm10_lag1,pm2.5_lag2,pm10_lag2,pm2.5_lag3,pm10_lag3,pm2.5_roll7,pm10_roll7
0,2022-08-10,31.2,71,10.0,46.541667,68.379167,78.018056,59.837500,86.254167,52.733333,75.850000,42.170833,61.029167,51.793296,75.070207
1,2022-08-11,27.9,81,17.3,27.416667,44.570833,45.694444,46.541667,68.379167,59.837500,86.254167,52.733333,75.850000,49.482143,72.411905
2,2022-08-12,29.9,70,15.9,31.612500,47.804167,53.633750,27.416667,44.570833,46.541667,68.379167,59.837500,86.254167,47.722619,70.133929
3,2022-08-13,30.0,72,13.7,32.675000,53.312500,55.369167,31.612500,47.804167,27.416667,44.570833,46.541667,68.379167,41.855357,62.457143
4,2022-08-14,28.4,83,7.1,53.270833,79.366667,89.009028,32.675000,53.312500,31.612500,47.804167,27.416667,44.570833,43.441071,65.076786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1231,2025-12-23,15.7,83,8.2,131.420833,134.104167,309.697404,219.558333,222.608333,205.012500,207.629167,148.308333,149.770833,149.836905,153.754167
1232,2025-12-24,15.3,72,9.4,57.645833,60.050000,96.154861,131.420833,134.104167,219.558333,222.608333,205.012500,207.629167,145.423810,149.148810
1233,2025-12-25,13.8,70,4.7,92.320833,94.820833,208.658750,57.645833,60.050000,131.420833,134.104167,219.558333,222.608333,139.458929,141.970833
1234,2025-12-26,14.0,75,4.0,129.504167,131.233333,308.237788,92.320833,94.820833,57.645833,60.050000,131.420833,134.104167,140.538690,142.888095


In [241]:
X=dfml[["pm2.5_lag1","pm10_lag1","pm2.5_lag2","pm10_lag2","pm2.5_lag3","pm10_lag3","pm2.5_roll7","pm10_roll7"]]
y_pm2_5=dfml["pm2.5"]
y_pm10=dfml["pm10"]

In [242]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor

In [243]:
cv=TimeSeriesSplit(n_splits=5)

In [244]:
params={
    "n_estimators":[100,200,300],
    "max_depth":[10,15,20],
    "min_samples_leaf":[1,2,5]    
}

In [245]:
def tune_rfreg(X,y,params):
    best_score_mae =np.inf
    best_score_mse =np.inf
    best_prams=None
    for n in params["n_estimators"]:
        for depth in params["max_depth"]:
            for leaf in params["min_samples_leaf"]:
                cv_mae=[]
                cv_mse=[]
                cv_r2=[]
                for train,val in cv.split(X):
                    X_train,X_val=X.iloc[train],X.iloc[val]
                    y_train,y_val=y.iloc[train],y.iloc[val]
                    model = RandomForestRegressor(
                        n_estimators=n,
                        max_depth=depth,
                        min_samples_leaf=leaf,
                        random_state=42,
                        n_jobs=-1
                    )
                    model.fit(X_train,y_train)
                    pred=model.predict(X_val)
                    cv_mae.append(mean_absolute_error(y_val,pred))
                    cv_mse.append(mean_squared_error(y_val,pred))
                    cv_r2.append(r2_score(y_val,pred))
                mean_cv_mae=np.mean(cv_mae)
                mean_cv_mse=np.mean(cv_mse)
                mean_cv_r2=np.mean(cv_r2)
                if mean_cv_mae<best_score_mae:
                    best_score_mae=mean_cv_mae
                if mean_cv_mse<best_score_mse:
                    best_score_mse=mean_cv_mse
                best_score_r2=mean_cv_r2
                best_params={
                    "n_estimators":n,
                    "max_depth":depth,
                    "min_samples_leaf":leaf
                }
    return best_params,best_score_mae,best_score_mse,best_score_r2

In [246]:
#tunning 2.5
best_params2_5,best_score_mae2_5,best_score_mse2_5,best_score_r2_2_5=tune_rfreg(X,y_pm2_5,params)
print(f"best_params2_5 = {best_params2_5}")
print(f"best_score_mae2_5 = {best_score_mae2_5}")
print(f"best_score_mse2_5 = {best_score_mse2_5}")
print(f"best_score_r2_2_5 = {best_score_r2_2_5}")

best_params2_5 = {'n_estimators': 300, 'max_depth': 20, 'min_samples_leaf': 5}
best_score_mae2_5 = 14.124873755167911
best_score_mse2_5 = 392.0657835443279
best_score_r2_2_5 = 0.4497244404912008


In [247]:
#tunning 10
best_params10,best_score_mae10,best_score_mse10,best_score_r2_10=tune_rfreg(X,y_pm10,params)
print(f"best_params10 = {best_params10}")
print(f"best_score_mae10 = {best_score_mae10}")
print(f"best_score_mse10 = {best_score_mse10}")
print(f"best_score_r2_10 = {best_score_r2_10}")

best_params10 = {'n_estimators': 300, 'max_depth': 20, 'min_samples_leaf': 5}
best_score_mae10 = 38.94085336427152
best_score_mse10 = 5468.765241218202
best_score_r2_10 = 0.5597378605694969


In [253]:
df_ml=df.dropna().reset_index(drop=True)
features = ["temperature", "humidity", "wind_speed","pm2.5_lag1", "pm2.5_lag2", "pm2.5_lag3","pm10_lag1", "pm10_lag2", "pm10_lag3","pm2.5_roll7", "pm10_roll7"]
X=df_ml[features]
y_pm2_5=df_ml["pm2.5"]
y_pm10=df_ml["pm10"]

In [254]:
from sklearn.ensemble import RandomForestRegressor

rf_pm25 = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_pm10 = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_pm25.fit(X, y_pm2_5)
rf_pm10.fit(X, y_pm10)

,n_estimators,300
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [259]:
from datetime import timedelta
def forecast_10_days(df_ml,rf_pm25,rf_pm10,df2_5,df10):
    df_forecast=df_ml.copy()
    result=[]
    for _ in range(10):
        last_row = df_forecast.iloc[-1]
        X_next = pd.DataFrame([last_row[features]], columns=features)
        pm25_pred=rf_pm25.predict(X_next)[0]
        pm10_pred=rf_pm10.predict(X_next)[0]
        aqi25=aqi(pm25_pred,df2_5)
        aqi10=aqi(pm10_pred,df10)
        aqi_final=max(aqi25,aqi10)
        next_date=last_row["date"]+timedelta(days=1)
        result.append({
            "date":next_date,
            "pm2.5":pm25_pred,
            "pm10":pm10_pred,
            "AQI":aqi_final
        })
        new_row=last_row.copy()
        new_row["date"]=next_date
        new_row["pm2.5"]=pm25_pred
        new_row["pm10"]=pm10_pred
        new_row["pm2.5_lag1"] = pm25_pred
        new_row["pm2.5_lag2"] = last_row["pm2.5_lag1"]
        new_row["pm2.5_lag3"] = last_row["pm2.5_lag2"]
        new_row["pm10_lag1"] = pm10_pred
        new_row["pm10_lag2"] = last_row["pm10_lag1"]
        new_row["pm10_lag3"] = last_row["pm10_lag2"]
        new_row["pm2.5_roll7"] =(df_forecast["pm2.5"].tail(6).mean() + pm25_pred) / 7
        new_row["pm10_roll7"] =(df_forecast["pm10"].tail(6).mean() + pm10_pred) / 7
        df_forecast=pd.concat([df_forecast,new_row.to_frame().T],ignore_index=True)
    return pd.DataFrame(result)    

In [260]:
forecast_df = forecast_10_days(df_ml,rf_pm25,rf_pm10,df2_5,df10)
forecast_df


,date,pm2.5,pm10,AQI
0,2025-12-28,140.385686,150.906217,316.524484
1,2025-12-29,131.322119,161.212927,309.622229
2,2025-12-30,119.499211,162.274387,298.347395
3,2025-12-31,118.793671,160.795557,296.019116
4,2026-01-01,119.186865,161.989502,297.316655
5,2026-01-02,117.838954,161.875320,292.868548
6,2026-01-03,117.584800,162.382108,292.029840
7,2026-01-04,118.000079,162.478883,293.400260
8,2026-01-05,118.067719,162.404175,293.623473
9,2026-01-06,118.067719,162.008946,293.623473


In [261]:
import joblib
joblib.dump(rf_pm25, "rf_pm25.pkl")
joblib.dump(rf_pm10, "rf_pm10.pkl")

['rf_pm10.pkl']

In [262]:
joblib.dump(features, "features.pkl")
joblib.dump(df2_5, "aqi_pm25_table.pkl")
joblib.dump(df10, "aqi_pm10_table.pkl")


['aqi_pm10_table.pkl']

In [264]:
rf_pm25 = joblib.load("rf_pm25.pkl")
rf_pm10 = joblib.load("rf_pm10.pkl")
features = joblib.load("features.pkl")